In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:

val_predictions_mph = np.load(
    "../results/urbanflow_v2_validation_predictions_mph.npy"
)

val_targets_mph = np.load(
    "../results/validation_targets_mph.npy"
)

validation_target_mask = np.load(
    "../results/validation_target_mask.npy"
)

print("Prediction shape:", val_predictions_mph.shape)
print("Target shape:", val_targets_mph.shape)
print("Mask shape:", validation_target_mask.shape)

Prediction shape: (3410, 207)
Target shape: (3410, 207)
Mask shape: (3410, 207)


In [3]:
def classify_congestion(speed):

    if speed <= 20:
        return "Severe Congestion"

    elif speed <= 40:
        return "Heavy Congestion"

    elif speed <= 60:
        return "Moderate Traffic"

    else:
        return "Free Flow"


class_labels = [
    "Severe Congestion",
    "Heavy Congestion",
    "Moderate Traffic",
    "Free Flow"
]

In [4]:
predicted_categories = np.vectorize(
    classify_congestion
)(val_predictions_mph)

actual_categories = np.vectorize(
    classify_congestion
)(val_targets_mph)

print("Predicted categories:", predicted_categories.shape)
print("Actual categories:", actual_categories.shape)

Predicted categories: (3410, 207)
Actual categories: (3410, 207)


In [5]:
valid_predicted_categories = predicted_categories[
    validation_target_mask
]

valid_actual_categories = actual_categories[
    validation_target_mask
]

print(
    "Valid observations:",
    len(valid_actual_categories)
)

Valid observations: 664844


In [6]:
accuracy = accuracy_score(
    valid_actual_categories,
    valid_predicted_categories
)

weighted_precision = precision_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="weighted",
    zero_division=0
)

macro_f1 = f1_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="macro",
    zero_division=0
)

print(f"Accuracy          : {accuracy:.4f}")
print(f"Weighted Precision: {weighted_precision:.4f}")
print(f"Weighted Recall   : {weighted_recall:.4f}")
print(f"Weighted F1       : {weighted_f1:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")

Accuracy          : 0.8189
Weighted Precision: 0.8248
Weighted Recall   : 0.8189
Weighted F1       : 0.8194
Macro F1          : 0.6717


In [7]:
report = classification_report(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print(report_df)

                   precision    recall  f1-score        support
Severe Congestion   0.763881  0.380466  0.507942   20467.000000
Heavy Congestion    0.613010  0.629635  0.621212   48895.000000
Moderate Traffic    0.616575  0.692362  0.652275  138884.000000
Free Flow           0.913598  0.897343  0.905398  456598.000000
accuracy            0.818923  0.818923  0.818923       0.818923
macro avg           0.726766  0.649951  0.671706  664844.000000
weighted avg        0.824836  0.818923  0.819385  664844.000000


In [8]:
report_df.to_csv(
    "../results/congestion_classification_report.csv"
)

print("Classification report saved.")

Classification report saved.


In [9]:
cm = confusion_matrix(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels
)

cm_df = pd.DataFrame(
    cm,
    index=class_labels,
    columns=class_labels
)

print(cm_df)

                   Severe Congestion  Heavy Congestion  Moderate Traffic  \
Severe Congestion               7787              9525              2400   
Heavy Congestion                1829             30786             13773   
Moderate Traffic                 367              6872             96158   
Free Flow                        211              3038             43624   

                   Free Flow  
Severe Congestion        755  
Heavy Congestion        2507  
Moderate Traffic       35487  
Free Flow             409725  


In [10]:
cm_df.to_csv(
    "../results/congestion_confusion_matrix.csv"
)

print("Confusion matrix saved.")

Confusion matrix saved.


In [11]:
actual_category_counts = pd.Series(
    valid_actual_categories
).value_counts()

predicted_category_counts = pd.Series(
    valid_predicted_categories
).value_counts()

category_distribution = pd.DataFrame({
    "actual_count": actual_category_counts,
    "predicted_count": predicted_category_counts
}).fillna(0).astype(int)

category_distribution["actual_percentage"] = (
    category_distribution["actual_count"]
    / len(valid_actual_categories)
    * 100
)

category_distribution["predicted_percentage"] = (
    category_distribution["predicted_count"]
    / len(valid_predicted_categories)
    * 100
)

category_distribution = category_distribution.reindex(
    class_labels
)

print(category_distribution)

                   actual_count  predicted_count  actual_percentage  \
Severe Congestion         20467            10194           3.078467   
Heavy Congestion          48895            50221           7.354357   
Moderate Traffic         138884           155955          20.889712   
Free Flow                456598           448474          68.677464   

                   predicted_percentage  
Severe Congestion              1.533292  
Heavy Congestion               7.553802  
Moderate Traffic              23.457382  
Free Flow                     67.455523  


In [12]:
category_distribution.to_csv(
    "../results/congestion_category_distribution.csv"
)

print("Category distribution saved.")

Category distribution saved.


In [13]:
category_to_number = {
    "Severe Congestion": 0,
    "Heavy Congestion": 1,
    "Moderate Traffic": 2,
    "Free Flow": 3
}

actual_numeric = np.array([
    category_to_number[x]
    for x in valid_actual_categories
])

predicted_numeric = np.array([
    category_to_number[x]
    for x in valid_predicted_categories
])

category_distance = np.abs(
    actual_numeric - predicted_numeric
)

print(
    "Exact category accuracy:",
    np.mean(category_distance == 0)
)

print(
    "One-category error:",
    np.mean(category_distance == 1)
)

print(
    "Two-or-more category error:",
    np.mean(category_distance >= 2)
)

print(
    "Mean category distance:",
    category_distance.mean()
)

Exact category accuracy: 0.8189229353051242
One-category error: 0.16712191130550927
Two-or-more category error: 0.013955153389366528
Mean category distance: 0.1964851905108567


In [14]:
category_error_results = pd.DataFrame({
    "metric": [
        "exact_category_accuracy",
        "one_category_error",
        "two_or_more_category_error",
        "mean_category_distance"
    ],
    "value": [
        np.mean(category_distance == 0),
        np.mean(category_distance == 1),
        np.mean(category_distance >= 2),
        category_distance.mean()
    ]
})

category_error_results.to_csv(
    "../results/congestion_category_error_analysis.csv",
    index=False
)

print("Category error analysis saved successfully.")

Category error analysis saved successfully.


## Conclusion

UrbanFlow's continuous traffic-speed predictions were converted into
four project-defined congestion categories using speed thresholds.

On 664,844 valid validation observations, the congestion interpretation
achieved:

- Accuracy: 81.89%
- Weighted F1-score: 81.94%
- Macro F1-score: 67.17%

Free-flow conditions were identified more consistently than severe
congestion conditions. Severe congestion had lower recall, indicating
that the model tends to underestimate some low-speed traffic conditions.

Category-distance analysis showed that 81.89% of predictions were
exactly correct, 16.71% were one category away, and only 1.40% were
two or more categories away.

The congestion categories are an interpretation layer applied to the
continuous speed predictions produced by UrbanFlow; they are not the
output of a separately trained classification model.